# Week 1, Lab 4 — A minimal ReAct loop

**ReAct** = Reason, Act, Observe, repeat until the model says it is done.

Frameworks hide this loop. You should be able to write it in ~40 lines.


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


## 1. Setup


In [3]:
WEEK = 'Week 1'
LAB = 'Lab 4 — ReAct loop'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 1 / Lab 4 — ReAct loop
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. The loop


In [5]:
STOP = "FINAL"
TOOLS = {
    "calculator": calculator,
    "lookup_fact": lookup_fact,
    "today_date": today_date,
}

SYSTEM = """You are a ReAct agent. Each turn, output ONLY JSON as one of:
{"thought": "...", "name": "<tool>", "arguments": {...}}
{"thought": "...", "final": "<answer to the user>"}

Tools: calculator(expression), lookup_fact(topic), today_date()
Never invent tool results. After you have enough observations, set final.
"""

def run_agent(question: str, max_steps: int = 6) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    for step in range(1, max_steps + 1):
        reply = local_chat(messages, max_new_tokens=160, temperature=0.1)
        print(f"\n--- step {step} ---\n{reply}")
        obj = extract_json_object(reply) or {}
        if "final" in obj:
            return str(obj["final"])
        name = obj.get("name")
        args = obj.get("arguments") or {}
        if name in TOOLS:
            result = TOOLS[name](**args) if args else TOOLS[name]()
        else:
            result = f"unknown tool {name!r}"
        print("OBS:", result)
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content": f"OBSERVATION: {result}"})
    return "Stopped: max steps reached."

print(run_agent("What is 19*21, and what is LangGraph?"))


Loading Hugging Face model Qwen/Qwen2.5-0.5B-Instruct on CPU ...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=160) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- step 1 ---
{"thought":"The expression '19*21' can be evaluated using the calculator function.","final":"LangGraph is not a real-world concept or entity that exists in the world."}
LangGraph is not a real-world concept or entity that exists in the world.


## 3. Failure modes to notice

Small models may: skip JSON, call the same tool twice, or 'final' too early. That is expected. Later weeks add retries, graphs, and guardrails because of this.

## 4. Exercise

1. Cap `max_steps` at 2 and describe what is lost.
2. Add a `scratchpad` list in Python (not in the prompt) and print it after the run.
3. Compare this loop to Week 4's LangGraph router — same idea, better structure.

**Next:** `lab5_mini_project.ipynb` — research-and-summarize without a framework.
